# Deploying Iris-detection model using Vertex AI


### Dataset

This notebook uses R.A. Fisher's Iris dataset, a small and popular dataset for machine learning experiments. Each instance has four numerical features, which are different measurements of a flower, and a target label that
categorizes the flower into: **Iris setosa**, **Iris versicolour** and **Iris virginica**.

This notebook uses [a version of the Iris dataset available in the
scikit-learn library](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html#sklearn.datasets.load_iris).

## Get started

### Install Vertex AI SDK for Python and other required packages



In [ ]:
# Vertex SDK for Python
! pip3 install --upgrade --quiet  google-cloud-aiplatform

### Set Google Cloud project information 
Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [4]:
PROJECT_ID = "eternal-cycling-463617-f7"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}

### Create a Cloud Storage bucket

Create a storage bucket to store intermediate artifacts such as datasets.

In [5]:
BUCKET_URI = f"gs://mlops-course-eternal-cycling-463617-f7-unique"  # @param {type:"string"}

**If your bucket doesn't already exist**: Run the following cell to create your Cloud Storage bucket.

In [6]:
! gsutil mb -l {LOCATION} -p {PROJECT_ID} {BUCKET_URI}

Creating gs://mlops-course-eternal-cycling-463617-f7-unique/...
ServiceException: 409 A Cloud Storage bucket named 'mlops-course-eternal-cycling-463617-f7-unique' already exists. Try another name. Bucket names must be globally unique across all Google Cloud projects, including those outside of your organization.


### Initialize Vertex AI SDK for Python

To get started using Vertex AI, you must have an existing Google Cloud project and [enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com). 

In [7]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET_URI)

### Import the required libraries

In [8]:
import os
import sys

### Configure resource names

Set a name for the following parameters:

`MODEL_ARTIFACT_DIR` - Folder directory path to your model artifacts within a Cloud Storage bucket, for example: "my-models/fraud-detection/trial-4"

`REPOSITORY` - Name of the Artifact Repository to create or use.

`IMAGE` - Name of the container image that is pushed to the repository.

`MODEL_DISPLAY_NAME` - Display name of Vertex AI model resource.

In [9]:
MODEL_ARTIFACT_DIR = "my-models/iris-classifier-week-1"  # @param {type:"string"}
REPOSITORY = "iris-classifier-repo"  # @param {type:"string"}
IMAGE = "iris-classifier-img"  # @param {type:"string"}
MODEL_DISPLAY_NAME = "iris-classifier"  # @param {type:"string"}

# Set the defaults if no names were specified
if MODEL_ARTIFACT_DIR == "[your-artifact-directory]":
    MODEL_ARTIFACT_DIR = "custom-container-prediction-model"

if REPOSITORY == "[your-repository-name]":
    REPOSITORY = "custom-container-prediction"

if IMAGE == "[your-image-name]":
    IMAGE = "sklearn-fastapi-server"

if MODEL_DISPLAY_NAME == "[your-model-display-name]":
    MODEL_DISPLAY_NAME = "sklearn-custom-container"

## Training and Logging
Training the models with GridSearch and then logging the best best pareameters of different models in the MLFlow

In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics
import mlflow
from mlflow.models import infer_signature
from mlflow import MlflowClient
import pickle
import joblib

In [11]:
# Load dataset
data = pd.read_csv('data/iris.csv')
X = data[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]
y = data['species']

In [12]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=42
)

In [13]:
# MLflow setup
mlflow.set_tracking_uri("http://127.0.0.1:8100")
client = MlflowClient(mlflow.get_tracking_uri())
all_experiments = client.search_experiments()
print(mlflow.get_tracking_uri())
mlflow.set_experiment("IRIS Grid Search")

http://127.0.0.1:8100


<Experiment: artifact_location='mlflow-artifacts:/124013209429601352', creation_time=1751786542849, experiment_id='124013209429601352', last_update_time=1751786542849, lifecycle_stage='active', name='IRIS Grid Search', tags={}>

In [14]:
# Define model configs
models = {
    "DecisionTree": {
        "model": DecisionTreeClassifier(random_state=42),
        "param_grid": {
            "max_depth": [4, 5],
            "min_samples_leaf": [1, 3],
            "min_samples_split": [2, 3]
        }
    },
    "RandomForest": {
        "model": RandomForestClassifier(random_state=42),
        "param_grid": {
            "n_estimators": [15, 30],
            "max_depth": [3, 5],
            "min_samples_split": [2,3]
        }
    }
}

# Store model results
model_results = []

# Loop through models and log with MLflow
for model_name, config in models.items():
    print(f"\nTraining and tuning: {model_name}")
    
    # Grid search
    grid = GridSearchCV(config["model"], config["param_grid"], cv=5, scoring='accuracy')
    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    best_params = grid.best_params_
    predictions = best_model.predict(X_test)
    accuracy = metrics.accuracy_score(y_test, predictions)

    # saving the model
    joblib.dump(best_model, "artifacts/model.joblib")

    # Log in MLflow
    with mlflow.start_run(run_name=model_name):
        mlflow.log_params(best_params)
        mlflow.log_metric("accuracy", accuracy)
        mlflow.set_tag("model_type", model_name)

        # Log model
        signature = infer_signature(X_train, best_model.predict(X_train))
        model_info = mlflow.sklearn.log_model(
            sk_model=best_model,
            artifact_path="iris_model",
            input_example=X_train.iloc[:5],
            signature=signature,
            registered_model_name="IRIS-Classifier-Search"
        )

        print(f"{model_name} logged with accuracy: {accuracy:.4f}")
        model_results.append((model_name, best_model, accuracy, model_info.model_uri))

# Compare and report best model
best = max(model_results, key=lambda x: x[2])  # highest accuracy
print(f"\n✅ Best Model: {best[0]} with Accuracy: {best[2]:.4f}")
print(f"Model URI: {best[3]}")



Training and tuning: DecisionTree


2025/07/06 07:40:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'IRIS-Classifier-Search' already exists. Creating a new version of this model...
2025/07/06 07:40:35 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: IRIS-Classifier-Search, version 3
Created version '3' of model 'IRIS-Classifier-Search'.


DecisionTree logged with accuracy: 0.9833
🏃 View run DecisionTree at: http://127.0.0.1:8100/#/experiments/124013209429601352/runs/54451a73bf5441fab5381a08687bc200
🧪 View experiment at: http://127.0.0.1:8100/#/experiments/124013209429601352

Training and tuning: RandomForest


2025/07/06 07:40:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'IRIS-Classifier-Search' already exists. Creating a new version of this model...
2025/07/06 07:40:40 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: IRIS-Classifier-Search, version 4


RandomForest logged with accuracy: 0.9333
🏃 View run RandomForest at: http://127.0.0.1:8100/#/experiments/124013209429601352/runs/0507c4d5d56746adbf49cb2709a4826f
🧪 View experiment at: http://127.0.0.1:8100/#/experiments/124013209429601352

✅ Best Model: DecisionTree with Accuracy: 0.9833
Model URI: models:/m-09ea7436721d4915b12647b7598e2a43


Created version '4' of model 'IRIS-Classifier-Search'.


In [17]:
!gsutil cp artifacts/model.joblib {BUCKET_URI}/{MODEL_ARTIFACT_DIR}/

Copying file://artifacts/model.joblib [Content-Type=application/octet-stream]...
/ [1 files][  2.5 KiB/  2.5 KiB]                                                
Operation completed over 1 objects/2.5 KiB.                                      
